# Tool Call Validation

This notebook demonstrates how to implement **tool call guardrails** using the Strands Agents SDK's `BeforeToolCallEvent` hook.

Tool call guardrails inspect tool names and arguments **before execution**, allowing you to:
- Enforce **allowlists** (only listed tools can run)
- Enforce **blocklists** (specific tools are blocked)
- Validate **arguments** for dangerous inputs (sensitive paths, dangerous commands)
- Log all tool call decisions for audit

**Key concept:** Use `BeforeToolCallEvent` via a `HookProvider` to intercept tool calls. Access `event.tool_use` dict with keys `name`, `toolUseId`, `input`. Set `event.cancel_tool = "reason"` to block a tool call.

## Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
import logging
from typing import Optional

from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.DEBUG, format="%(asctime)s [%(levelname)s] %(name)s - %(message)s", datefmt="%H:%M:%S")

## Pattern 1: Allowlist Enforcement

Only tools whose names appear in the allowlist are permitted to execute. All other tools are blocked with a descriptive reason.

In [ ]:
def allowlist_validate(event, allowed_tools: list[str]) -> None:
    """Validate a tool call against an allowlist."""
    tool_name = event.tool_use.get("name", "")

    if tool_name not in allowed_tools:
        reason = f"Tool '{tool_name}' is not permitted. Allowed tools: {allowed_tools}"
        event.cancel_tool = reason
        logger.warning(f"[TOOL GUARDRAIL] BLOCKED tool='{tool_name}' reason='not in allowlist'")
    else:
        logger.debug(f"[TOOL GUARDRAIL] ALLOWED tool='{tool_name}' reason='in allowlist'")

## Pattern 2: Blocklist Enforcement

Tools in the blocklist are blocked. All other tools are allowed. This is the inverse of allowlist — useful when you want to restrict a few dangerous tools but allow everything else.

In [ ]:
def blocklist_validate(event, blocked_tools: list[str]) -> None:
    """Validate a tool call against a blocklist."""
    tool_name = event.tool_use.get("name", "")

    if tool_name in blocked_tools:
        reason = f"Tool '{tool_name}' is explicitly blocked."
        event.cancel_tool = reason
        logger.warning(f"[TOOL GUARDRAIL] BLOCKED tool='{tool_name}' reason='in blocklist'")
    else:
        logger.debug(f"[TOOL GUARDRAIL] ALLOWED tool='{tool_name}' reason='not in blocklist'")

## Pattern 3: Argument Validation

Inspects tool arguments to detect dangerous patterns:
- File operations targeting sensitive paths (`/etc/passwd`, `~/.ssh/`, `.env`)
- Shell commands containing dangerous patterns (`rm -rf`, `sudo`, `curl | sh`)

This provides defense-in-depth beyond just checking tool names.

In [ ]:
# Sensitive paths that should never be accessed
SENSITIVE_PATHS = [
    "/etc/passwd", "/etc/shadow", "/root/",
    "~/.ssh/", "~/.aws/credentials", ".env",
]

# Dangerous shell patterns
DANGEROUS_COMMANDS = [
    "rm -rf", "mkfs", "dd if=", "> /dev/",
    "chmod 777", "curl | sh", "wget | sh",
]


def argument_validate(
    event,
    sensitive_paths: Optional[list[str]] = None,
    dangerous_commands: Optional[list[str]] = None,
) -> None:
    """Validate tool arguments for dangerous patterns."""
    paths = sensitive_paths or SENSITIVE_PATHS
    commands = dangerous_commands or DANGEROUS_COMMANDS

    tool_name = event.tool_use.get("name", "")
    tool_input = event.tool_use.get("input", {})

    for arg_name, arg_value in tool_input.items():
        if not isinstance(arg_value, str):
            continue

        for sensitive_path in paths:
            if sensitive_path in arg_value:
                reason = (
                    f"Tool '{tool_name}' argument '{arg_name}' references "
                    f"sensitive path: '{sensitive_path}'"
                )
                event.cancel_tool = reason
                logger.warning(f"[TOOL GUARDRAIL] BLOCKED tool='{tool_name}' reason='sensitive path'")
                return

        for dangerous_cmd in commands:
            if dangerous_cmd in arg_value:
                reason = (
                    f"Tool '{tool_name}' argument '{arg_name}' contains "
                    f"dangerous command pattern: '{dangerous_cmd}'"
                )
                event.cancel_tool = reason
                logger.warning(f"[TOOL GUARDRAIL] BLOCKED tool='{tool_name}' reason='dangerous command'")
                return

    logger.debug(f"[TOOL GUARDRAIL] ALLOWED tool='{tool_name}' reason='arguments validated'")

## Pattern 4: Combined Guardrail with Audit Logging

Combines all three patterns into a single comprehensive guardrail:
1. Allowlist check (if configured)
2. Blocklist check (if configured)
3. Argument validation

In [ ]:
def combined_tool_validate(
    event,
    allowed_tools: Optional[list[str]] = None,
    blocked_tools: Optional[list[str]] = None,
    sensitive_paths: Optional[list[str]] = None,
    dangerous_commands: Optional[list[str]] = None,
) -> None:
    """Comprehensive tool call validation combining multiple strategies."""
    paths = sensitive_paths or SENSITIVE_PATHS
    commands = dangerous_commands or DANGEROUS_COMMANDS

    tool_name = event.tool_use.get("name", "")
    tool_input = event.tool_use.get("input", {})

    # Step 1: Allowlist check
    if allowed_tools is not None and tool_name not in allowed_tools:
        reason = f"Tool '{tool_name}' is not in the allowed tools list."
        event.cancel_tool = reason
        _audit_log(tool_name, "BLOCKED", "not in allowlist", tool_input)
        return

    # Step 2: Blocklist check
    if blocked_tools is not None and tool_name in blocked_tools:
        reason = f"Tool '{tool_name}' is explicitly blocked."
        event.cancel_tool = reason
        _audit_log(tool_name, "BLOCKED", "in blocklist", tool_input)
        return

    # Step 3: Argument validation
    for arg_name, arg_value in tool_input.items():
        if not isinstance(arg_value, str):
            continue
        for sensitive_path in paths:
            if sensitive_path in arg_value:
                reason = f"Tool '{tool_name}' blocked: sensitive path '{sensitive_path}'"
                event.cancel_tool = reason
                _audit_log(tool_name, "BLOCKED", f"sensitive path: {sensitive_path}", tool_input)
                return
        for dangerous_cmd in commands:
            if dangerous_cmd in arg_value:
                reason = f"Tool '{tool_name}' blocked: dangerous pattern '{dangerous_cmd}'"
                event.cancel_tool = reason
                _audit_log(tool_name, "BLOCKED", f"dangerous command: {dangerous_cmd}", tool_input)
                return

    _audit_log(tool_name, "ALLOWED", "all checks passed", tool_input)


def _audit_log(tool_name, decision, reason, tool_input):
    input_summary = {k: str(v)[:50] for k, v in tool_input.items()} if tool_input else {}
    log_msg = f"[TOOL AUDIT] tool='{tool_name}' decision={decision} reason='{reason}' input={input_summary}"
    if decision == "BLOCKED":
        logger.warning(log_msg)
    else:
        logger.info(log_msg)

## Demo: Testing Each Pattern with Mock Events

In [ ]:
class MockToolEvent:
    """Mock BeforeToolCallEvent for demonstration."""
    def __init__(self, tool_name: str, tool_input: Optional[dict] = None):
        self.tool_use = {"name": tool_name, "toolUseId": "test-123", "input": tool_input or {}}
        self.cancel_tool = None


# --- Pattern 1: Allowlist ---
print("--- Pattern 1: Allowlist Enforcement ---")
print("Only 'calculator' and 'file_reader' are allowed.\n")

allowed = ["calculator", "file_reader"]

event = MockToolEvent("calculator", {"expression": "2 + 2"})
allowlist_validate(event, allowed)
print(f"  Tool: 'calculator'       -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")

event = MockToolEvent("shell_execute", {"command": "ls"})
allowlist_validate(event, allowed)
print(f"  Tool: 'shell_execute'    -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

In [ ]:
# --- Pattern 2: Blocklist ---
print("--- Pattern 2: Blocklist Enforcement ---")
print("'shell_execute' and 'file_delete' are blocked.\n")

blocked = ["shell_execute", "file_delete"]

event = MockToolEvent("calculator", {"expression": "3 * 7"})
blocklist_validate(event, blocked)
print(f"  Tool: 'calculator'       -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")

event = MockToolEvent("shell_execute", {"command": "whoami"})
blocklist_validate(event, blocked)
print(f"  Tool: 'shell_execute'    -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

In [ ]:
# --- Pattern 3: Argument Validation ---
print("--- Pattern 3: Argument Validation ---")
print("Blocks tools that access sensitive paths or run dangerous commands.\n")

# Safe file read
event = MockToolEvent("file_reader", {"path": "/tmp/report.txt"})
argument_validate(event)
print(f"  file_reader('/tmp/report.txt')     -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")

# Sensitive path
event = MockToolEvent("file_reader", {"path": "/etc/passwd"})
argument_validate(event)
print(f"  file_reader('/etc/passwd')         -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

# Dangerous command
event = MockToolEvent("shell_execute", {"command": "rm -rf /tmp/data"})
argument_validate(event)
print(f"  shell_execute('rm -rf /tmp/data')  -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

# Safe command
event = MockToolEvent("shell_execute", {"command": "echo hello"})
argument_validate(event)
print(f"  shell_execute('echo hello')        -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")

In [ ]:
# --- Pattern 4: Combined Guardrail ---
print("--- Pattern 4: Combined Guardrail ---")
print("Allowlist + argument validation together.\n")

# Allowed tool with safe arguments
event = MockToolEvent("calculator", {"expression": "100 / 4"})
combined_tool_validate(event, allowed_tools=["calculator", "file_reader", "shell_execute"])
print(f"  calculator('100 / 4')              -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")

# Tool not in allowlist
event = MockToolEvent("web_search", {"query": "test"})
combined_tool_validate(event, allowed_tools=["calculator", "file_reader", "shell_execute"])
print(f"  web_search('test')                 -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

# Allowed tool but dangerous arguments
event = MockToolEvent("shell_execute", {"command": "rm -rf /"})
combined_tool_validate(event, allowed_tools=["calculator", "file_reader", "shell_execute"])
print(f"  shell_execute('rm -rf /')          -> {'ALLOWED' if event.cancel_tool is None else 'BLOCKED'}")
print(f"        Reason: {event.cancel_tool}")

## HookProvider: Wrapping Tool Guardrails for Agent Registration

In strands-agents 1.40.0, hooks are registered via `HookProvider` classes.

In [ ]:
class ToolGuardrailHook(HookProvider):
    """HookProvider that validates tool calls using the combined guardrail."""

    def __init__(
        self,
        allowed_tools: Optional[list[str]] = None,
        blocked_tools: Optional[list[str]] = None,
        sensitive_paths: Optional[list[str]] = None,
        dangerous_commands: Optional[list[str]] = None,
    ):
        self.allowed_tools = allowed_tools
        self.blocked_tools = blocked_tools
        self.sensitive_paths = sensitive_paths
        self.dangerous_commands = dangerous_commands

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self._validate_tool)

    def _validate_tool(self, event: BeforeToolCallEvent) -> None:
        combined_tool_validate(
            event,
            allowed_tools=self.allowed_tools,
            blocked_tools=self.blocked_tools,
            sensitive_paths=self.sensitive_paths,
            dangerous_commands=self.dangerous_commands,
        )


print("ToolGuardrailHook defined successfully.")
print("Register with: Agent(hooks=[ToolGuardrailHook(allowed_tools=[...])])")

## Attaching to a Live Agent

In [ ]:
try:
    from strands import Agent, tool
    from strands.models.bedrock import BedrockModel

    @tool
    def calculator(expression: str) -> str:
        """Evaluate a math expression."""
        return str(eval(expression))

    @tool
    def file_reader(path: str) -> str:
        """Read a file from disk."""
        return f"Contents of {path}"

    @tool
    def shell_execute(command: str) -> str:
        """Execute a shell command."""
        return f"Executed: {command}"

    model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0")

    tool_hook = ToolGuardrailHook(
        allowed_tools=["calculator", "file_reader"],
        sensitive_paths=SENSITIVE_PATHS,
        dangerous_commands=DANGEROUS_COMMANDS,
    )

    agent = Agent(
        model=model,
        system_prompt="You are a helpful assistant with access to tools.",
        tools=[calculator, file_reader, shell_execute],
        hooks=[tool_hook],
    )

    print("Agent created with tool call guardrail.")
    print("Testing: What is 25 * 4?")
    response = agent("What is 25 * 4?")
    print(f"  Response: {response}")

except Exception as e:
    print(f"Skipping live agent demo: {e}")
    print("(This is expected if no model provider is configured)")

## Summary

In this notebook you learned four tool call validation patterns:
1. **Allowlist** — only listed tools can run
2. **Blocklist** — specific tools are blocked, all others allowed
3. **Argument validation** — inspect arguments for dangerous patterns
4. **Combined guardrail** — all patterns together with audit logging

And how to wrap them in a `HookProvider` for agent registration.

**Next Steps:** See `06_error_handling.ipynb` to learn about fail-open vs fail-closed error handling patterns.